# AI-Based Credit Default Risk Prediction
### Step-by-Step Machine Learning Project for Beginners

This notebook teaches how to build a Machine Learning model that predicts whether a credit card customer might fail to make their next payment.

- **Target = 0**: Customer will pay on time (No Default).
- **Target = 1**: Customer will fail to pay (Default).

## Step 1: Load the Dataset
We import Pandas and NumPy, read the Excel dataset, and inspect its rows and columns.

In [ ]:
# Import basic data science libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Path to the dataset excel file
data_path = os.path.join('..', 'data', 'dataset.xls')

# The excel file has title info on row 0, so header starts at row 1
df = pd.read_excel(data_path, header=1)

# Rename target column to standard name
if 'default payment next month' in df.columns:
    df.rename(columns={'default payment next month': 'default.payment.next.month'}, inplace=True)
if 'PAY_0' in df.columns:
    df.rename(columns={'PAY_0': 'PAY_1'}, inplace=True)

print('Dataset dimensions (Rows, Columns):', df.shape)
print('Missing values total:', df.isnull().sum().sum())
print('Duplicate rows total:', df.duplicated().sum())
df.head()

## Step 2: Understand the Dataset (Exploratory Data Analysis)
We check how many customers defaulted and visualize key financial patterns.

In [ ]:
# Count default vs non-default customers
target = 'default.payment.next.month'
counts = df[target].value_counts()
percentages = df[target].value_counts(normalize=True) * 100

print(f'Total Customers: {len(df)}')
print(f'No Default (0): {counts[0]} ({percentages[0]:.2f}%)')
print(f'Default (1): {counts[1]} ({percentages[1]:.2f}%)')

# Graph: Default vs No Default
plt.figure(figsize=(6, 4))
sns.countplot(x=target, data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Credit Card Default Distribution')
plt.xticks([0, 1], ['No Default (0)', 'Default (1)'])
plt.xlabel('Customer Status')
plt.ylabel('Number of Customers')
plt.show()

## Step 3: Clean the Data
- We remove `ID` because customer IDs are just random serial numbers, not financial features.
- We clean unknown categories in `EDUCATION` and `MARRIAGE`.
- We separate input features ($X$) and the output target ($y$).

In [ ]:
# Clean EDUCATION (0, 4, 5, 6 mapped to 4 = Others)
df['EDUCATION'] = df['EDUCATION'].apply(lambda x: 4 if x in [0, 4, 5, 6] else x)

# Clean MARRIAGE (0 mapped to 3 = Others)
df['MARRIAGE'] = df['MARRIAGE'].apply(lambda x: 3 if x == 0 else x)

# Separate inputs (X) and output (y)
feature_columns = [col for col in df.columns if col not in ['ID', 'default.payment.next.month']]
X = df[feature_columns].copy()
y = df['default.payment.next.month'].copy()

print('Input features count:', X.shape[1])
print('Features list:', feature_columns)

## Step 4: Split the Data into Training and Testing Sets
- 80% is used for training the model.
- 20% is held out for testing.
- We use `stratify=y` to ensure both train and test have the exact same ratio of defaults.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale numerical features so all variables are on the same scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Training records:', len(X_train))
print('Testing records:', len(X_test))

## Step 5: Handle Imbalanced Data
Only 22% of customers defaulted, while 78% paid on time. If the model predicts 'No Default' all the time, it would achieve 78% accuracy but miss every default! We compare SMOTE and Class Weighting.

In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE generates synthetic examples of default customers on training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print('Original train balance:', y_train.value_counts().to_dict())
print('SMOTE train balance:', pd.Series(y_train_smote).value_counts().to_dict())

## Step 6 & 7: Train and Compare 5 ML Models
1. **Logistic Regression**: Simple statistical formula estimating default probability.
2. **Decision Tree**: Sequence of yes/no decision questions.
3. **Random Forest**: Ensemble of multiple decision trees voting together.
4. **Support Vector Machine**: Draws a decision boundary separating default from non-default.
5. **XGBoost**: Gradient boosted trees that learn sequentially from past errors.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'Support Vector Machine': CalibratedClassifierCV(LinearSVC(dual=False, random_state=42, max_iter=2000)),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

comparison_data = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    
    comparison_data.append({
        'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4),
        'Recall': round(rec, 4), 'F1-Score': round(f1, 4), 'ROC-AUC': round(roc, 4), 'PR-AUC': round(pr, 4)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df

## Step 8: Find the Best Decision Threshold using Business Cost Matrix
Default probability is between 0.0 and 1.0.
- If a customer defaults and we missed them (**False Negative**), the bank loses their unpaid debt (Assumed Cost = $500).
- If a customer is flagged as high risk but would have paid (**False Positive**), the bank spends customer support time checking (Assumed Cost = $50).

We test thresholds from 0.10 to 0.90 to find the threshold with the lowest total financial cost.

In [ ]:
from sklearn.metrics import confusion_matrix

best_rf = models['Random Forest']
test_probs = best_rf.predict_proba(X_test_scaled)[:, 1]

cost_fn = 500  # Cost of missed default (Assumed)
cost_fp = 50   # Cost of false alert review (Assumed)

thresholds = np.arange(0.10, 0.95, 0.05)
cost_records = []

for t in thresholds:
    pred_binary = (test_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_binary).ravel()
    total_cost = (fn * cost_fn) + (fp * cost_fp)
    cost_records.append({'Threshold': round(t, 2), 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn, 'Cost': total_cost})

cost_df = pd.DataFrame(cost_records)
best_row = cost_df.loc[cost_df['Cost'].idxmin()]
print(f'Optimal Threshold: {best_row["Threshold"]} with Lowest Cost: ${best_row["Cost"]:,}')
cost_df

## Step 9: Save the Model and Preprocessing Steps
We save the trained Random Forest model and the scaler using Joblib so our FastAPI website can use it without retraining.

In [ ]:
import joblib

models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

# Save model
joblib.dump(best_rf, os.path.join(models_dir, 'model.pkl'))

# Save preprocessing
prep_dict = {
    'scaler': scaler,
    'feature_names': feature_columns,
    'optimal_threshold': float(best_row['Threshold']),
    'cost_fn': cost_fn,
    'cost_fp': cost_fp,
    'threshold_records': cost_records
}
joblib.dump(prep_dict, os.path.join(models_dir, 'preprocessing.pkl'))
print('Model and Preprocessing pipeline successfully saved to ../models/!')